[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/02_preprocessing/02_preprocessing_solutions.ipynb)

# 02. 전처리 — 연습 문제 해설

[02_preprocessing.ipynb](02_preprocessing.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와 해설입니다.
**먼저 직접 시도해본 뒤** 참고하세요.

본문과 같은 순서입니다. **문제 1~3은 1부(택시), 문제 4~6은 2부(타이타닉)** 범위입니다.

> **읽는 법** — 셀은 위에서부터 순서대로 실행해야 합니다(`Shift + Enter`). 실행 결과는 저장되어
> 있지 않으니 직접 실행해야 표와 그래프가 나타납니다. 맨 위의 **준비 셀들을 먼저 실행한 뒤**
> 원하는 문제로 건너뛰면 됩니다. 해설에 적힌 숫자는 실행하면 나오는 값입니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀

아래 셀들은 본문과 같은 준비 코드입니다. **내용을 이해할 필요 없이 그대로 실행**하면 됩니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

두 데이터를 한 번에 불러옵니다. **문제 1~3은 `trips`, 문제 4~6은 `titanic`** 을 씁니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

titanic = sns.load_dataset("titanic")

print("trips  :", trips.shape)
print("titanic:", titanic.shape)

---

# 1부 — 택시 (`trips`)

## 문제 1. `duration`에 IQR 기준을 적용하면?

이 문제의 핵심은 **"IQR이 이상치라고 지목했다고 해서 정말 이상치인 것은 아니다"** 입니다.

In [ ]:
q1 = trips["duration"].quantile(0.25)
q3 = trips["duration"].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1          : {q1:8.4f} 분")
print(f"Q3          : {q3:8.4f} 분")
print(f"IQR         : {iqr:8.4f}")
print(f"lower_fence : {lower_fence:8.4f} 분")
print(f"upper_fence : {upper_fence:8.4f} 분")

outlier = (trips["duration"] < lower_fence) | (trips["duration"] > upper_fence)
print(f"\n이상치: {outlier.sum()}건 ({outlier.mean() * 100:.1f}%)")

IQR이 이상치로 지목한 358건이 **정말 잘못된 값인지** 보려면, 남는 행과 비교해봐야 합니다.

In [ ]:
# 제거되는 행과 남는 행을 비교
removed = trips[outlier]
kept = trips[~outlier]

compare = pd.DataFrame({
    "제거되는 358건": removed[["distance", "duration", "fare"]].median(),
    "남는 6075건": kept[["distance", "duration", "fare"]].median(),
}).round(2)
compare

승차 자치구 분포를 보면 이 행들의 정체가 더 분명해집니다.

In [ ]:
print("제거되는 행의 승차 자치구")
print(removed["pickup_borough"].value_counts())
print()
print("전체 대비 비율")
print((removed["pickup_borough"].value_counts(normalize=True) * 100).round(1))

그래프로도 확인합니다. 오른쪽 산점도에서 **이상치로 지목된 점들이 정상 데이터와 같은 직선 위에
이어져 있는지**를 보세요. 기록 오류라면 직선에서 벗어나 있어야 합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

labeled = trips.assign(구분=outlier.map({True: "이상치로 판정(358건)", False: "정상(6075건)"}))

sns.boxplot(data=labeled, x="구분", y="distance", ax=axes[0])
axes[0].set_title("이동 거리")

sns.scatterplot(data=labeled, x="distance", y="duration", hue="구분", alpha=0.4, ax=axes[1])
axes[1].set_title("거리 - 시간")

plt.tight_layout()
plt.show()

**해설 — 제거하면 안 됩니다**

`upper_fence`는 36.54분이고, 이보다 오래 걸린 운행 358건(5.6%)이 이상치로 지목됩니다.
그런데 이 행들을 열어보면 성격이 분명합니다.

| | 제거되는 358건 | 남는 6,075건 |
|---|---|---|
| `distance` 중앙값 | **12.70 마일** | 1.55 마일 |
| `duration` 중앙값 | 46.12 분 | 10.28 분 |
| `fare` 중앙값 | 41.92 달러 | 9.00 달러 |

**거리가 8배 깁니다.** 오래 걸린 게 아니라 **멀리 간 것**입니다. 산점도를 보면 이 점들이
정상 데이터와 같은 직선 위에 자연스럽게 이어져 있습니다. 기록 오류라면 직선에서 벗어나 있어야 합니다.

승차 자치구를 보면 더 분명해집니다. 전체에서는 Queens가 10%인데 **이 358건에서는 38%**입니다.
JFK·LaGuardia 공항이 모두 Queens에 있습니다. 즉 **공항 노선**입니다.

지우면 어떻게 될까요? 우리의 목표는 이동 시간 예측입니다.

- 공항 노선은 **예측이 가장 중요한 구간**입니다. 5분짜리 운행은 틀려도 타격이 없지만
  50분짜리 운행 예측이 틀리면 비행기를 놓칩니다
- 모델은 학습 데이터에서 본 적 없는 구간을 예측하지 못합니다. 긴 운행을 다 지우고 학습시키면
  **정작 필요한 상황에서 쓸모없는 모델**이 됩니다

**결론**: `duration`에는 IQR 기준을 적용하지 않습니다. 본문에서 `speed >= 60`(9.4마일을 7초에 이동)만
제거한 것은 그것이 **물리적으로 불가능한 기록 오류**이기 때문입니다.

> **IQR 기준은 "정규분포에서 벗어난 값"을 찾는 도구지, "잘못된 값"을 찾는 도구가 아닙니다.**
> 오른쪽으로 치우친 분포에서는 정상적인 큰 값들이 대량으로 걸립니다. 걸린 행이
> 실제로 어떤 행인지 확인하지 않고 지우는 것이 가장 위험합니다.

## 문제 2. 에러는 없지만 결과가 잘못된 코드

**문제로 주어진 코드**

```python
scaler = StandardScaler()
X_all = scaler.fit_transform(X_reg)              # ← 전체 데이터로 fit
X_train, X_valid, y_train, y_valid = train_test_split(
    X_all, y_reg, test_size=0.2, random_state=42
)
```

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()   # 이상치 제거
    df = df.drop(columns=["pickup", "dropoff",                     # 시각 자체는 weekday/hour로 대체
                          "pickup_zone", "dropoff_zone",           # 범주가 200개 이상이라 제외
                          "speed",                                 # duration으로 계산한 값 → 정답 누출
                          "total"])                                # fare+tip+tolls의 합 → 중복
    df = df.dropna()                                               # 결측치 행 제거
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)                           # 범주형 → 0/1
    X = df.drop(columns="duration")
    y = df["duration"]
    return X, y


def prepare_titanic(raw):
    """[분류] 타이타닉 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()  # 이상치 제거
    df = df.drop(columns=["alive",                                   # survived와 같은 정보 → 정답 누출
                          "class", "embark_town",                    # pclass/embarked와 중복
                          "deck",                                    # 결측치가 77%
                          "adult_male"])                             # who와 중복
    df = df.dropna()
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)
    X = df.drop(columns="survived")
    y = df["survived"]
    return X, y

잘못된 방법과 올바른 방법을 나란히 실행해, **스케일링 기준(평균)이 실제로 달라지는지** 확인합니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_reg, y_reg = prepare_trips(trips)

# ❌ 잘못된 방법: 나누기 전에 전체 데이터로 fit
bad_scaler = StandardScaler()
X_all = bad_scaler.fit_transform(X_reg)
bad_train, bad_valid, y_tr, y_va = train_test_split(
    X_all, y_reg, test_size=0.2, random_state=42
)

# ✅ 올바른 방법: 먼저 나누고, 학습 데이터로만 fit
X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
good_scaler = StandardScaler()
good_train = good_scaler.fit_transform(X_train)
good_valid = good_scaler.transform(X_valid)

print("두 방법의 스케일링 기준(평균) 차이")
diff = np.abs(bad_scaler.mean_ - good_scaler.mean_)
print(f"  최대 차이: {diff.max():.6f}")
print()
print("검증 데이터 최댓값")
print(f"  잘못된 방법: {bad_valid.max():.4f}")
print(f"  올바른 방법: {good_valid.max():.4f}")

**해설 — 무엇이 문제인가**

`scaler.fit(X_reg)`는 **전체 6,336건의 평균과 표준편차를 계산**합니다. 그런데 그 안에는
나중에 검증용으로 떼어낼 1,268건이 포함되어 있습니다.

즉 **"한 번도 보지 못한 데이터"여야 할 검증 데이터의 정보가 학습 데이터 변환에 섞여 들어갑니다.**
이것이 **데이터 누출(data leakage)** 입니다.

**고치는 법**

```python
# 1. 먼저 나눈다
X_train, X_valid, y_train, y_valid = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# 2. 학습 데이터로만 fit
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)          # transform만!
```

**왜 발견하기 어려운가**

- **에러가 나지 않습니다.** 코드는 정상적으로 끝까지 실행됩니다
- **차이가 작습니다.** 위 실험에서 평균의 차이는 소수점 아래에서 나타납니다. 데이터가 많고
  분포가 고르면 전체 통계와 학습 데이터 통계가 비슷하기 때문입니다
- **점수가 살짝 좋아집니다.** 그래서 "잘 됐네"하고 넘어가기 쉽습니다

**그런데 왜 문제 삼는가**

1. **데이터가 적거나 이상치가 있으면 차이가 커집니다.** 검증 데이터에만 있는 극단값 하나가
   전체 평균을 흔들면, 학습 데이터 변환까지 영향을 받습니다
2. **실전에서 재현할 수 없는 상황입니다.** 서비스에서는 예측 요청이 한 건씩 들어옵니다.
   미래 데이터의 평균을 미리 알 방법이 없습니다. 검증 점수가 실제 성능보다 높게 나오면
   **배포 후에 성능이 떨어지는 이유를 알 수 없게 됩니다**
3. **교차 검증에서는 훨씬 심각해집니다.** fold를 나눌 때마다 누출이 반복 누적됩니다

> 이 문제를 구조적으로 막는 방법이 scikit-learn의 `Pipeline`입니다.
> ```python
> from sklearn.pipeline import make_pipeline
> model = make_pipeline(StandardScaler(), RandomForestRegressor())
> model.fit(X_train, y_train)   # 내부에서 알아서 fit_transform / transform 구분
> ```
> 전처리와 모델을 하나로 묶어두면 `fit`을 잘못 호출할 여지 자체가 없어집니다.
> 03번 노트북의 교차 검증에서 이 방식을 씁니다.

## 문제 3. `pickup_zone`(194개)을 살려서 쓰려면

In [ ]:
work = trips[(trips["duration"] > 0) & (trips["speed"] < 60)].copy()
work = work.drop(columns=["pickup", "dropoff", "speed", "total"]).dropna()

print("pickup_zone 고유값:", work["pickup_zone"].nunique(), "개")
print()
print("상위 10개")
print(work["pickup_zone"].value_counts().head(10))

상위 몇 개를 남기느냐에 따라 결과가 어떻게 달라지는지 봅니다.

In [ ]:
# 상위 N개만 남기고 나머지를 "Other"로 묶기
for n in [10, 20, 30]:
    top_zones = work["pickup_zone"].value_counts().head(n).index
    grouped = work["pickup_zone"].where(work["pickup_zone"].isin(top_zones), "Other")
    print(f"상위 {n:2d}개 유지 -> 고유값 {grouped.nunique():2d}개, "
          f"'Other'로 묶이는 비율 {(grouped == 'Other').mean() * 100:.1f}%")

상위 20개로 실제 인코딩해서 **세 방식의 컬럼 수**를 비교합니다.

In [ ]:
# 상위 20개로 실제 인코딩해서 컬럼 수 비교
def group_rare(series, n):
    top = series.value_counts().head(n).index
    return series.where(series.isin(top), "Other")


reduced = work.copy()
reduced["pickup_zone"] = group_rare(reduced["pickup_zone"], 20)
reduced["dropoff_zone"] = group_rare(reduced["dropoff_zone"], 20)

obj_cols = work.select_dtypes(include="object").columns.tolist()

full = pd.get_dummies(work, columns=obj_cols, drop_first=True)
trimmed = pd.get_dummies(reduced, columns=obj_cols, drop_first=True)
borough_only = pd.get_dummies(work.drop(columns=["pickup_zone", "dropoff_zone"]),
                              columns=["color", "payment", "pickup_borough", "dropoff_borough"],
                              drop_first=True)

print(f"zone 전체 사용      : {full.shape[1]:3d}개 컬럼")
print(f"zone 상위 20개 + Other: {trimmed.shape[1]:3d}개 컬럼")
print(f"zone 삭제 (본문 방식) : {borough_only.shape[1]:3d}개 컬럼")

**해설 — 고유값이 많은 범주형을 다루는 방법들**

| 방법 | 컬럼 수 | 특징 |
|---|---|---|
| 그대로 인코딩 | **412** | 정보 손실 없음. 하지만 대부분의 컬럼이 거의 항상 0 |
| **상위 20개 + Other** | **57** | 자주 나오는 지역은 구분하고 나머지는 묶음 |
| zone 삭제, borough만 | **17** | 가장 단순. 본문의 선택 |

승·하차 zone 두 컬럼을 각각 상위 20개로 줄이는 것만으로 **412개가 57개**가 됩니다.
그러면서도 맨해튼 주요 지역은 여전히 개별 컬럼으로 구분됩니다.

**① 희소 범주 묶기 (Other)** — 위에서 구현한 방법입니다.

```python
top = series.value_counts().head(20).index
series.where(series.isin(top), "Other")
```

`where`는 **조건이 참인 곳은 그대로 두고, 거짓인 곳을 두 번째 인자로 바꿉니다.**
(`mask`는 반대로 동작합니다.) 상위 20개로도 전체의 51%를 커버합니다.

**② 상위 계층으로 대체** — 본문에서 쓴 방법입니다. `pickup_zone`(194개) 대신
`pickup_borough`(5개)를 씁니다. 위계가 있는 범주(동 → 구 → 시)에서 가장 간단하고 효과적입니다.

**③ 빈도 인코딩 (frequency encoding)** — 범주를 그 범주의 등장 횟수로 바꿉니다.

```python
freq = work["pickup_zone"].value_counts(normalize=True)
work["pickup_zone_freq"] = work["pickup_zone"].map(freq)
```

컬럼이 **하나만** 늘어납니다. "번화가인가 한적한 곳인가"가 숫자로 표현되는 셈입니다.

**④ 타깃 인코딩 (target encoding)** — 범주별 타깃 평균으로 바꿉니다. 예: 각 zone의 평균 이동 시간.
강력하지만 **정답을 써서 피처를 만드는 것이라 데이터 누출 위험이 큽니다.**
반드시 학습 데이터에서만 평균을 계산하고 교차 검증 안에서 처리해야 합니다.

**어느 것을 고를까**

컬럼이 400개가 되어도 트리 모델은 견디지만, **대부분 값이 0인 컬럼(희소 행렬)은
분할 후보만 늘리고 실제 기여는 거의 없습니다.** 학습 시간만 길어집니다.

이 시리즈가 zone을 버린 이유는 **`borough`가 이미 지역 정보를 담고 있어서**입니다.
같은 정보를 더 거친 단위로 가진 컬럼이 있다면 그쪽을 쓰는 것이 거의 항상 낫습니다.

---

# 2부 — 타이타닉 (`titanic`)

## 문제 4. `age` 결측치를 세 가지 방법으로 채우기

In [ ]:
age = titanic["age"]

fill_all = age.fillna(age.median())
fill_pclass = titanic.groupby("pclass")["age"].transform(lambda s: s.fillna(s.median()))
fill_pclass_sex = titanic.groupby(["pclass", "sex"])["age"].transform(lambda s: s.fillna(s.median()))

summary = pd.DataFrame({
    "평균": [age.mean(), fill_all.mean(), fill_pclass.mean(), fill_pclass_sex.mean()],
    "표준편차": [age.std(), fill_all.std(), fill_pclass.std(), fill_pclass_sex.std()],
}, index=["원본(결측 제외)", "① 전체 중앙값", "② pclass별", "③ pclass+sex별"]).round(2)
summary

각 방법이 실제로 **어떤 값을 채워 넣는지** 확인합니다.

In [ ]:
print("① 전체 중앙값:", age.median())
print()
print("② pclass별 중앙값")
print(titanic.groupby("pclass")["age"].median())
print()
print("③ pclass + sex별 중앙값")
print(titanic.groupby(["pclass", "sex"])["age"].median())

네 분포를 나란히 그려, 대체 때문에 생기는 **뾰족한 막대**가 방법마다 어떻게 달라지는지 봅니다.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for ax, (name, v) in zip(axes, [
    ("원본 (결측 제외)", age.dropna()),
    ("① 전체 중앙값", fill_all),
    ("② pclass별", fill_pclass),
    ("③ pclass+sex별", fill_pclass_sex),
]):
    sns.histplot(v, bins=30, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("age")

plt.tight_layout()
plt.show()

**해설**

| 방법 | 평균 | 표준편차 |
|---|---|---|
| 원본 (결측 제외) | 29.70 | **14.53** |
| ① 전체 중앙값 (28.0) | 29.36 | 13.02 |
| ② `pclass`별 | 29.07 | 13.24 |
| ③ `pclass`+`sex`별 | 29.11 | **13.30** |

**세 방법 모두 표준편차를 줄입니다.** 없던 값을 기존 값으로 채우니 당연합니다.
다만 **그룹을 잘게 나눌수록 원본(14.53)에 가까워집니다.**

- ①은 177명 전부를 28.0세로 만듭니다 → 히스토그램에 뾰족한 막대 하나
- ②는 등급별로 37 / 29 / 24세를 씁니다 → 막대가 세 개로 분산
- ③은 여섯 그룹으로 더 잘게 나눕니다 → 가장 자연스러움

`pclass`별 중앙값 차이(1등급 37세 vs 3등급 24세)가 **13세**나 됩니다. 이 차이를 무시하고
전부 28세로 채우면 1등급 승객은 실제보다 어리게, 3등급은 나이 들게 기록되는 셈입니다.

**그렇다고 그룹을 무한정 잘게 나눌 수는 없습니다.** 그룹이 작아지면 중앙값 자체가 불안정해지고,
극단적으로 그룹에 결측치만 있으면 채울 값 자체가 없어집니다(`NaN`이 그대로 남습니다).
보통 **그룹당 최소 수십 개 이상**의 데이터가 있는 선에서 멈춥니다.

> `transform`을 쓴 이유: 그룹별로 계산한 결과를 **원래 행 순서 그대로** 돌려주기 때문입니다.
> `agg`나 `apply`는 그룹 단위로 축약된 결과를 주기 때문에 컬럼에 그대로 대입할 수 없습니다.

## 문제 5. `embarked` 결측치를 최빈값으로

In [ ]:
print("결측치:", titanic["embarked"].isnull().sum(), "건")
print()
print("값의 분포:")
print(titanic["embarked"].value_counts())
print()

mode_result = titanic["embarked"].mode()
print("mode()의 반환 타입:", type(mode_result).__name__)
print(mode_result)

최빈값으로 채운 뒤 결측이 사라졌는지, 분포가 얼마나 바뀌었는지 확인합니다.

In [ ]:
embarked_filled = titanic["embarked"].fillna(titanic["embarked"].mode()[0])

print("채우기 전 결측:", titanic["embarked"].isnull().sum())
print("채운 후 결측  :", embarked_filled.isnull().sum())
print()
print(embarked_filled.value_counts())

**해설 — `mode()`에 `[0]`이 필요한 이유**

`mean()`이나 `median()`은 **스칼라 값 하나**를 반환합니다. 평균과 중앙값은 언제나 하나뿐이니까요.

하지만 **최빈값은 여러 개일 수 있습니다.** 값 A와 B가 똑같이 100번씩 나타나면 최빈값은 둘 다입니다.
그래서 pandas의 `mode()`는 **Series를 반환**합니다.

```python
titanic["embarked"].mode()
# 0    S
# dtype: object       ← 값 하나짜리 Series
```

`fillna()`에 Series를 그대로 넘기면 의도와 다르게 동작하므로, **`[0]`으로 첫 번째 값을 꺼내야** 합니다.

```python
df["embarked"].fillna(df["embarked"].mode()[0])   # ✅
df["embarked"].fillna(df["embarked"].mode())      # ❌ 채워지지 않음
```

여기서는 `S`(Southampton)가 644건으로 압도적이라 최빈값이 하나뿐입니다. 결측 2건을 채워도
분포는 사실상 변하지 않습니다. **결측 비율이 0.2%라면 `dropna()`로 지워도 무방**하고,
본문의 파이프라인은 그렇게 처리합니다.

> 최빈값이 여러 개일 때 무조건 `[0]`을 쓰는 것이 최선인지는 생각해볼 문제입니다.
> `mode()`는 값을 정렬해서 반환하므로 `[0]`은 "동점 중 사전순으로 앞선 것"이 됩니다.
> 근거가 있는 선택은 아니지만, 동점이 생길 정도라면 어느 쪽을 골라도 큰 차이가 없는 상황이기도 합니다.

## 문제 6. `drop_first`를 끄면?

In [ ]:
q1, q3 = titanic["fare"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

base = titanic[(titanic["fare"] >= lower) & (titanic["fare"] <= upper)].copy()
base = base.drop(columns=["alive", "class", "embark_town", "deck", "adult_male"]).dropna()

enc_true = pd.get_dummies(base, columns=["sex", "embarked", "who"], drop_first=True)
enc_false = pd.get_dummies(base, columns=["sex", "embarked", "who"], drop_first=False)

print(f"drop_first=True  : {enc_true.shape[1]}개 컬럼")
print(f"drop_first=False : {enc_false.shape[1]}개 컬럼")
print()
print("추가되는 컬럼:", sorted(set(enc_false.columns) - set(enc_true.columns)))

되살아난 컬럼이 정말 **잉여**인지, 즉 나머지 컬럼으로 완전히 결정되는지 직접 확인합니다.

In [ ]:
# 추가된 컬럼이 정말 "잉여" 정보인지 확인
print("sex_female + sex_male 의 합이 항상 1인가?")
print((enc_false["sex_female"] + enc_false["sex_male"]).value_counts())
print()
print("embarked 세 컬럼의 합이 항상 1인가?")
print((enc_false["embarked_C"] + enc_false["embarked_Q"] + enc_false["embarked_S"]).value_counts())

**해설**

12개 → 15개로 **3개 늘어납니다.** 인코딩 대상이 `sex`, `embarked`, `who` 세 개이므로
각각 첫 컬럼이 하나씩 되살아납니다.

| 컬럼 | 원래 범주 |
|---|---|
| `sex_female` | sex: female / male |
| `embarked_C` | embarked: C / Q / S |
| `who_child` | who: child / man / woman |

`sex_female + sex_male`이 **모든 행에서 정확히 1**이고, `embarked` 세 컬럼의 합도 항상 1입니다.
즉 **하나를 빼도 정보가 전혀 손실되지 않습니다.** `sex_male=0`이 곧 `sex_female=1`이니까요.

**트리 모델이라면 `drop_first=False`가 낫습니다.**

- 트리는 다중공선성의 영향을 받지 않습니다. 계수를 푸는 것이 아니라 컬럼마다 독립적으로
  "이 값보다 큰가?"를 물을 뿐이기 때문입니다
- **변수중요도를 읽기 편해집니다.** `drop_first=True`로 `embarked_C`를 빼면, "C에서 탄 것의 효과"가
  `embarked_Q`와 `embarked_S`에 흩어져 해석이 어려워집니다
- 컬럼이 3개 늘어나는 비용은 무시할 만합니다

**회귀·신경망이라면 `drop_first=True`** 를 씁니다. 선형 회귀에서 완전한 다중공선성은
계수 계산 자체를 불안정하게 만듭니다.

이 시리즈가 `drop_first=True`로 통일한 이유는 03(트리)과 04(신경망)에서 **같은 데이터를
쓰기 위해서**입니다. 트리만 다룬다면 `False`가 더 나은 선택입니다.

---

다음 노트북: [03_tree_models.ipynb](../03_tree_models/03_tree_models.ipynb)